[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/traceopt-ai/traceml/blob/main/notebooks/huggingface_dataloading_bottleneck.ipynb)

# Find a data-loading bottleneck with Hugging Face Trainer

A training job can keep making progress while its GPU repeatedly waits for the next batch. This notebook answers a practical question:

> Can a better DataLoader configuration reduce input wait and keep the GPU busy?

You will train the same Hugging Face ResNet-50 job twice on the 320px Imagenette dataset. TraceML diagnoses each run and shows whether the loader change improved training time.

The result is hardware-dependent. A machine with more CPU capacity may feed the GPU more effectively, while a smaller Colab runtime may remain input-bound after tuning.

## The comparison

| Profile | DataLoader workers | Pinned memory | Persistent workers |
|---|---:|---:|---:|
| Baseline | 0 | No | No |
| Optimized | Up to 4 | Yes | Yes |

The model, images, augmentation, batch size, seed, and optimizer-step count stay fixed. The optimized profile changes the loader settings as one practical configuration, so this experiment measures the profile as a whole rather than attributing the result to one setting.

TraceML records DataLoader wait separately from model compute. Trainer runtime shows the complete training loop, while `traceml compare` shows where the difference occurred.

## 1. Set the run length

Two hundred optimizer steps are enough to produce a stable diagnosis on a Colab T4.

In [ ]:
MAX_STEPS = 200
BATCH_SIZE = 32

## 2. Check the GPU

In Colab, select **Runtime > Change runtime type > T4 GPU**.

In [ ]:
import torch

!nvidia-smi -L
assert torch.cuda.is_available(), "Enable a GPU runtime before continuing."
print("Using:", torch.cuda.get_device_name(0))

GPU 0: Tesla T4 (UUID: GPU-2fe46d9c-5dcd-0ca2-9d3b-0f5b59944399)
Using: Tesla T4


## 3. Install dependencies

The Hugging Face integration is included in the `hf` TraceML extra. The training code still uses the standard `Trainer` and `TrainingArguments` APIs.

In [ ]:
%pip install -q -U "traceml-ai[hf]" transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.6/559.6 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 4.8 MB/s eta 0:00:00


## 4. Download the dataset

Imagenette contains real JPEG images, so decoding and augmentation create meaningful CPU work without requiring the full ImageNet dataset. This notebook downloads the 326 MiB 320px archive.

In [ ]:
import os

if not os.path.isdir("imagenette2-320/train"):
    !wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz
    !tar -xzf imagenette2-320.tgz

print("CPU cores available:", os.cpu_count())
print(
    "Training images:",
    sum(len(files) for _, _, files in os.walk("imagenette2-320/train")),
)

CPU cores available: 2
Training images: 9469


## 5. Define the training run

The script uses `microsoft/resnet-50`, the Imagenette training split, 224px random crops, and a batch size of 32. Each profile runs in a separate process through `traceml run`, which keeps model and CUDA state isolated.

### Add TraceML

The training code remains a standard `Trainer` run. TraceML needs two additions:

```python
traceml_hf.init()
callbacks=[traceml_hf.TraceMLTrainerCallback()]
```

The notebook writes the complete script below so it remains self-contained.

In [ ]:
%%writefile hf_train.py
"""Run one Hugging Face DataLoader configuration."""

import argparse
import json
import os
from pathlib import Path

import torch
from torch.utils.data import Dataset
from torchvision.datasets import ImageFolder
from torchvision.transforms import (
    Compose,
    Normalize,
    RandomHorizontalFlip,
    RandomResizedCrop,
    ToTensor,
)
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    DefaultDataCollator,
    Trainer,
    TrainingArguments,
    set_seed,
)

from traceml_ai.integrations import huggingface as traceml_hf

MODEL_ID = "microsoft/resnet-50"
SEED = 42


class ImagenetteForTrainer(Dataset):
    """Return the field names expected by AutoModelForImageClassification."""

    def __init__(self, root, transform):
        self.images = ImageFolder(root, transform=transform)
        self.classes = self.images.classes

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        pixel_values, label = self.images[index]
        return {"pixel_values": pixel_values, "labels": label}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--profile", choices=["baseline", "optimized"], required=True
    )
    parser.add_argument("--data-dir", default="imagenette2-320")
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--max-steps", type=int, default=200)
    args = parser.parse_args()

    set_seed(SEED)
    optimized = args.profile == "optimized"
    num_workers = min(4, os.cpu_count() or 2) if optimized else 0
    pin_memory = optimized
    persistent_workers = optimized and num_workers > 0
    output_dir = Path("outputs") / args.profile
    output_dir.mkdir(parents=True, exist_ok=True)

    image_processor = AutoImageProcessor.from_pretrained(MODEL_ID)
    crop_size = image_processor.size.get(
        "height", image_processor.size.get("shortest_edge", 224)
    )
    transform = Compose(
        [
            RandomResizedCrop(crop_size),
            RandomHorizontalFlip(),
            ToTensor(),
            Normalize(
                mean=image_processor.image_mean,
                std=image_processor.image_std,
            ),
        ]
    )
    train_dataset = ImagenetteForTrainer(
        os.path.join(args.data_dir, "train"), transform=transform
    )

    id2label = {index: name for index, name in enumerate(train_dataset.classes)}
    label2id = {name: index for index, name in id2label.items()}
    model = AutoModelForImageClassification.from_pretrained(
        MODEL_ID,
        num_labels=len(train_dataset.classes),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        per_device_train_batch_size=args.batch_size,
        max_steps=args.max_steps,
        learning_rate=1e-4,
        save_strategy="no",
        report_to="none",
        disable_tqdm=True,
        remove_unused_columns=False,
        dataloader_num_workers=num_workers,
        dataloader_pin_memory=pin_memory,
        dataloader_persistent_workers=persistent_workers,
        seed=SEED,
        data_seed=SEED,
    )

    print(
        f"[demo] profile={args.profile} workers={num_workers} "
        f"pin_memory={pin_memory} persistent_workers={persistent_workers} "
        f"batch_size={args.batch_size} max_steps={args.max_steps}",
        flush=True,
    )

    traceml_hf.init()
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=DefaultDataCollator(),
        callbacks=[traceml_hf.TraceMLTrainerCallback()],
    )
    train_result = trainer.train()
    metrics = {
        **train_result.metrics,
        "profile": args.profile,
        "dataloader_num_workers": num_workers,
        "dataloader_pin_memory": pin_memory,
        "dataloader_persistent_workers": persistent_workers,
        "optimizer_steps": train_result.global_step,
        "gpu_name": torch.cuda.get_device_name(0),
    }
    metrics_path = output_dir / "trainer_metrics.json"
    metrics_path.write_text(
        json.dumps(metrics, indent=2, sort_keys=True),
        encoding="utf-8",
    )


if __name__ == "__main__":
    main()

Writing hf_train.py


## 6. Run the configurations

### A. Baseline loader

With no worker processes, the training process decodes and augments every batch itself. TraceML reports `INPUT-BOUND` when that work leaves the GPU waiting.

In [ ]:
!traceml run \
    --mode summary \
    --logs-dir logs \
    --run-name hf_baseline \
    hf_train.py \
    --args \
    --profile baseline \
    --data-dir imagenette2-320 \
    --max-steps {MAX_STEPS} \
    --batch-size {BATCH_SIZE}

[TraceML] Starting aggregator on 127.0.0.1:29765 (connect=127.0.0.1, ui=summary, profile=run)
[TraceML] Launching TraceML aggregator: /usr/bin/python3 /usr/local/lib/python3.12/dist-packages/traceml_ai/aggregator/aggregator_main.py
[TraceML] Aggregator PID: 1531
[TraceML] Aggregator ready on 127.0.0.1:29765 (workers connect to 127.0.0.1:29765, session=hf_baseline, ui=summary). Press Ctrl+C to stop.
[TraceML] Aggregator ready.
[TraceML] Launching training process: /usr/bin/python3 -m torch.distributed.run --nnodes=1 --nproc_per_node=1 --node_rank=0 --master_addr=127.0.0.1 --master_port=29500 /usr/local/lib/python3.12/dist-packages/traceml_ai/runtime/executor.py -- --profile baseline --data-dir imagenette2-320 --max-steps 200 --batch-size 32
preprocessor_config.json: 100% 266/266 [00:00<00:00, 1.06MB/s]
config.json: 100% 69.6k/69.6k [00:00<00:00, 63.6MB/s]
[transformers] You passed `num_labels=10` which is incompatible to the `id2label` map of length `1000`.

model.safetensors: downloadi

### B. Optimized loader

This profile lets up to four CPU workers prepare batches ahead of time, uses pinned memory, and keeps the workers alive. Everything outside the DataLoader profile remains unchanged.

In [ ]:
!traceml run \
    --mode summary \
    --logs-dir logs \
    --run-name hf_optimized \
    hf_train.py \
    --args \
    --profile optimized \
    --data-dir imagenette2-320 \
    --max-steps {MAX_STEPS} \
    --batch-size {BATCH_SIZE}

[TraceML] Starting aggregator on 127.0.0.1:29765 (connect=127.0.0.1, ui=summary, profile=run)
[TraceML] Launching TraceML aggregator: /usr/bin/python3 /usr/local/lib/python3.12/dist-packages/traceml_ai/aggregator/aggregator_main.py
[TraceML] Aggregator PID: 2220
[TraceML] Aggregator ready on 127.0.0.1:29765 (workers connect to 127.0.0.1:29765, session=hf_optimized, ui=summary). Press Ctrl+C to stop.
[TraceML] Aggregator ready.
[TraceML] Launching training process: /usr/bin/python3 -m torch.distributed.run --nnodes=1 --nproc_per_node=1 --node_rank=0 --master_addr=127.0.0.1 --master_port=29500 /usr/local/lib/python3.12/dist-packages/traceml_ai/runtime/executor.py -- --profile optimized --data-dir imagenette2-320 --max-steps 200 --batch-size 32
[transformers] You passed `num_labels=10` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100% 320/320 [00:00<00:00, 5398.94it/s]
[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-50
Key  

## 7. Compare the runs

TraceML compares the saved summaries, explains the diagnosis change, and writes JSON for the results table.

In [ ]:
!traceml compare \
    logs/hf_baseline/final_summary.json \
    logs/hf_optimized/final_summary.json \
    --output=logs/hf_baseline_vs_optimized

+--------------------------------------------------------------------------------------+
|  TraceML Compare                                                                     |
+--------------------------------------------------------------------------------------+
|                                                                                      |
|  A: hf_baseline                                                                      |
|  B: hf_optimized                                                                     |
|  Delta: B - A                                                                        |
|  Primary diagnosis: INPUT-BOUND -> COMPUTE-BOUND (changed)                           |
|                                                                                      |
|  Verdict: IMPROVEMENT                                                                |
|  Why: GPU Step Time decreased by 25.3%.                                              |
|                    

## 8. Review the result

The table combines Trainer runtime with the TraceML measurements that explain the difference.

In [ ]:
import json
from pathlib import Path

import pandas as pd


def load_json(path):
    with Path(path).open(encoding="utf-8") as handle:
        return json.load(handle)


def compare_value(payload, section, metric, side):
    return (
        payload.get("sections", {})
        .get(section, {})
        .get("metrics", {})
        .get(metric, {})
        .get(side)
    )


def rounded(value, digits=2, scale=1.0):
    if value is None:
        return None
    return round(float(value) / scale, digits)


comparison = load_json("logs/hf_baseline_vs_optimized.json")


def result_row(profile, side):
    trainer = load_json(f"outputs/{profile}/trainer_metrics.json")
    return {
        "profile": profile,
        "workers": trainer.get("dataloader_num_workers"),
        "pinned memory": trainer.get("dataloader_pin_memory"),
        "persistent workers": trainer.get("dataloader_persistent_workers"),
        "Trainer runtime (s)": rounded(trainer.get("train_runtime")),
        "Trainer steps/s": rounded(
            trainer.get("train_steps_per_second"), digits=3
        ),
        "TraceML step (ms)": rounded(
            compare_value(comparison, "step_time", "step_time_ms", side)
        ),
        "input wait (ms)": rounded(
            compare_value(comparison, "step_time", "input_ms", side)
        ),
        "compute (ms)": rounded(
            compare_value(comparison, "step_time", "compute_ms", side)
        ),
        "peak reserved (GB)": rounded(
            compare_value(
                comparison,
                "step_memory",
                "peak_reserved_bytes",
                side,
            ),
            scale=1e9,
        ),
    }


pd.DataFrame([result_row("baseline", "lhs"), result_row("optimized", "rhs")])

,profile,workers,pinned memory,persistent workers,Trainer runtime (s),Trainer steps/s,TraceML step (ms),input wait (ms),compute (ms),peak reserved (GB)
0,baseline,0,False,False,89.61,2.232,442.01,129.12,304.46,3.33
1,optimized,2,True,True,66.65,3.001,329.99,2.02,319.14,3.33


### Example result from one Colab T4 run

| Profile | Trainer runtime | Trainer steps/s | TraceML step | Input wait | Compute | Peak reserved |
|---|---:|---:|---:|---:|---:|---:|
| Baseline | 87.62 s | 2.282 | 431.4 ms | 121.5 ms | 303.3 ms | 3.10 GB |
| Optimized | 66.07 s | 3.027 | 325.5 ms | 2.5 ms | 314.2 ms | 3.10 GB |

In this run, the diagnosis changed from **INPUT-BOUND** to **COMPUTE-BOUND**. Input wait fell by 98.0%, and both Trainer runtime and TraceML step time improved by about 24.5%. Peak reserved GPU memory did not change.

The exact values will vary with CPU capacity, storage, GPU type, and software versions. The important test is whether input wait and wall time improve together on the machine that will run the workload.

## Read the result

- Trainer runtime and steps per second show whether the complete training loop improved.
- TraceML step and input-wait time show whether data preparation caused the difference.
- Compute time should remain broadly similar because the model and batch size did not change.
- A missing or zero host-to-device value does not prove that transfers took no time; they may occur outside the traced callback window.

For benchmark numbers, run each profile three times in fresh runtimes and report the median. To identify which DataLoader setting matters, test workers, pinned memory, and persistent workers one at a time after this profile-level comparison.

## Use this in your own Trainer

This pattern is not specific to ResNet. In an existing Hugging Face script, call `traceml_hf.init()` once, add `TraceMLTrainerCallback`, and launch the script with `traceml run --mode summary`.

If TraceML reports `INPUT-BOUND`, change one loader setting at a time and keep it only when both wall time and input wait improve on your hardware.